# RL Trading Agent Colab Trainer

Open this notebook in VS Code with the Colab extension and run the cells from top to bottom.

This notebook is set up for the standalone `trading_agent/` package and trains on the combined India + US watchlists by default.

## 1. Mount Google Drive

This keeps your models, checkpoints, and cached sentiment data safe even if the Colab session disconnects.

In [26]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Move Into Your Repo

This notebook assumes your repo folder is stored in Google Drive as `/content/drive/MyDrive/Apex`.

If your folder name or location is different, change the path in the next cell.

In [27]:
import os
from pathlib import Path

candidate_roots = [
    Path('/content/drive/MyDrive/Apex'),
    Path('/content/drive/MyDrive/Apex-main'),
    Path('/content/Apex'),
]

repo_root = next((path for path in candidate_roots if (path / 'trading_agent').exists()), None)
if repo_root is None:
    raise FileNotFoundError(
        'Could not find the Apex repo. Put the full Apex folder in Google Drive or clone it into /content first.'
    )

os.chdir(repo_root)
print('Repo root:', repo_root)
!pwd
!ls

Repo root: /content/drive/MyDrive/Apex
/content/drive/MyDrive/Apex
 apex_dashboard.py	 colab_train.ipynb   __pycache__	  trading_agent
 apex_dual_state.json	 Dockerfile	     railway.toml
 apex.log		 env		     requirements.txt
 apex_trading_agent.py	'HOW TO USE.md'      supabase_setup.sql


## 3. Install Dependencies

In [28]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '-r',
    str(repo_root / 'trading_agent' / 'requirements.txt'),
])
print('Dependencies installed from', repo_root / 'trading_agent' / 'requirements.txt')

Dependencies installed from /content/drive/MyDrive/Apex/trading_agent/requirements.txt


## 4. Set Environment Variables

Paste your Finnhub key in the next cell before running it.

The training universe is already configured to use your combined India + US watchlists, so you do not need to set `RL_TICKERS` manually.

In [29]:
import os
from pathlib import Path

from dotenv import load_dotenv

env_path = repo_root / '.env'
storage_root = '/content/drive/MyDrive/trading_agent_runtime'

if not env_path.exists():
    env_path.write_text(
        '\n'.join([
            'FINNHUB_API_KEY=PASTE_YOUR_FINNHUB_KEY_HERE',
            'ENABLE_SENTIMENT=true',
            'RL_MARKET_UNIVERSE=all',
            'RL_CHECKPOINT_FREQ=25000',
            'RL_EVAL_FREQ=10000',
            '',
        ]),
        encoding='utf-8',
    )
    raise FileNotFoundError(f'Created {env_path}. Paste your Finnhub key there, then rerun this cell.')

load_dotenv(env_path, override=True)
os.environ.setdefault('ENABLE_SENTIMENT', 'true')
os.environ.setdefault('RL_MARKET_UNIVERSE', 'all')
os.environ.setdefault('RL_CHECKPOINT_FREQ', '25000')
os.environ.setdefault('RL_EVAL_FREQ', '10000')

assert os.environ.get('FINNHUB_API_KEY') not in {None, '', 'PASTE_YOUR_FINNHUB_KEY_HERE'}, (
    f'Set FINNHUB_API_KEY in {env_path} first.'
)
print('Loaded configuration from', env_path)
print('Environment configured for combined India + US watchlist training.')
print('Artifacts will be stored in', storage_root)

Loaded configuration from /content/drive/MyDrive/Apex/.env
Environment configured for combined India + US watchlist training.
Artifacts will be stored in /content/drive/MyDrive/trading_agent_runtime


## 5. Optional Quick Smoke Test

Run this first if you want a shorter training session before launching the full 500,000-timestep run.

In [30]:
import os
import subprocess
import sys

os.environ["RL_TOTAL_TIMESTEPS"] = "200000"

env = dict(os.environ)
env["PYTHONPATH"] = str(repo_root)

subprocess.check_call([
    sys.executable,
    "-m",
    "trading_agent.main",
    "--mode",
    "train",
    "--resume",
    "--storage-dir",
    "/content/drive/MyDrive/trading_agent_runtime",
], env=env)


KeyboardInterrupt: 

## 6. Full Training Run

This uses Google Drive-backed storage and resume-friendly checkpoints.

If the Colab session disconnects, reconnect and rerun this cell with `--resume`.

In [33]:
import os
import subprocess
import sys

env = dict(os.environ)
env["PYTHONPATH"] = str(repo_root)

subprocess.check_call([
    sys.executable,
    "-m",
    "trading_agent.main",
    "--mode",
    "train",
    "--resume",
    "--storage-dir",
    storage_root,
], env=env)


KeyboardInterrupt: 

## 7. Inspect Saved Artifacts

In [ ]:
from pathlib import Path

artifact_dir = Path(storage_root) / 'agent' / 'model'
print('Artifact directory:', artifact_dir)
for path in sorted(artifact_dir.glob('*')):
    print(path.name)

Artifact directory: /content/drive/MyDrive/trading_agent_runtime/agent/model


## Notes

- Default tickers are the combined `INDIA_WATCHLIST` and `US_WATCHLIST`.
- Checkpoints are stored under `trading_agent_runtime/agent/model/checkpoints/`.
- Best model is saved as `best_model.zip`.
- Latest model is saved as `latest_model.zip`.